In [2]:
# imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
import requests
import os
from bs4 import BeautifulSoup
import re

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

In [24]:
import io

cities = {
    'los-angeles': 'united-states/ca/los-angeles',
    'new-york-city': 'united-states/ny/new-york-city',
    'chicago': 'united-states/il/chicago',
    'seattle': 'united-states/wa/seattle',
    'austin': 'united-states/tx/austin'
}

for city, path in cities.items():
    try:
        urls = get_insideairbnb_url(path, rank=1)
        response = requests.get(urls['listings'])
        df = pd.read_csv(io.BytesIO(response.content), compression='gzip', 
                        usecols=['price', 'estimated_occupancy_l365d', 'neighbourhood_cleansed'])
        print(f"\n{city}:")
        print(f"  Date: {urls['date']}")
        print(f"  Listings: {len(df)}")
        print(f"  Price non-null: {df['price'].notna().sum()}")
        print(f"  Occupancy non-null: {df['estimated_occupancy_l365d'].notna().sum()}")
        print(f"  Neighbourhoods: {df['neighbourhood_cleansed'].nunique()}")
    except Exception as e:
        print(f"\n{city}: FAILED — {e}")

Using date: 2025-09-01

los-angeles:
  Date: 2025-09-01
  Listings: 45886
  Price non-null: 36819
  Occupancy non-null: 45886
  Neighbourhoods: 266
Using date: 2025-11-01

new-york-city:
  Date: 2025-11-01
  Listings: 36353
  Price non-null: 21415
  Occupancy non-null: 36353
  Neighbourhoods: 224
Using date: 2025-06-17

chicago:
  Date: 2025-06-17
  Listings: 8604
  Price non-null: 7681
  Occupancy non-null: 8604
  Neighbourhoods: 76
Using date: 2025-06-21

seattle:
  Date: 2025-06-21
  Listings: 6862
  Price non-null: 6227
  Occupancy non-null: 6862
  Neighbourhoods: 88
Using date: 2025-06-13

austin:
  Date: 2025-06-13
  Listings: 15187
  Price non-null: 10708
  Occupancy non-null: 15187
  Neighbourhoods: 44


In [25]:
for city, path in cities.items():
    os.makedirs(f'../data/raw/{city}', exist_ok=True)
    urls = get_insideairbnb_url(path, rank=1)
    
    for name, url in urls.items():
        if name == 'date':
            continue
        ext = '.geojson' if 'geojson' in url else '.csv.gz'
        filepath = f'../data/raw/{city}/{name}{ext}'
        print(f'Downloading {city}/{name}...')
        response = requests.get(url)
        with open(filepath, 'wb') as f:
            f.write(response.content)

print("\nAll cities downloaded")

Using date: 2025-09-01
Using date: 2025-11-01
Using date: 2025-06-17
Using date: 2025-06-21
Using date: 2025-06-13

All cities downloaded!
